<a href="https://colab.research.google.com/github/vaisshnavee1410/Clustering_Analysis.ipynb/blob/main/Clustering_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CLUSTERING ANALYSIS**

## **Understanding and Implementing K-Means, Hierarchical, and DBSCAN Algorithms:**

### **OBJECTIVE:**
The objective of this assignment is to introduce to various clustering algorithms, including K-Means, hierarchical, and DBSCAN, and provide hands-on experience in applying these techniques to a real-world dataset.

### **DATASETS:**

### **Data Preprocessing:**

* Preprocess the dataset to handle missing values, remove outliers, and scale the features if necessary.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

# Load the dataset (replace with your file if needed)
df = pd.read_excel("EastWestAirlines.xlsx", sheet_name="data")

# Step 1: Drop ID or non-relevant columns
df = df.drop(columns=["ID#"])

# Step 2: Handle missing values (if any)
# Check for missing values
if df.isnull().sum().sum() > 0:
    df = df.dropna()  # Or use df.fillna(method='ffill') to impute

# Step 3: Remove outliers using Z-score
z_scores = np.abs(zscore(df))
df_no_outliers = df[(z_scores < 3).all(axis=1)]

# Step 4: Feature scaling
scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_no_outliers)
df_scaled = pd.DataFrame(scaled_array, columns=df_no_outliers.columns)

# Result
print("Original shape:", df.shape)
print("After outlier removal:", df_no_outliers.shape)
print("Scaled data preview:")
print(df_scaled.head())

* Perform exploratory data analysis (EDA) to gain insights into the distribution of data and identify potential clusters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Use seaborn styling
sns.set(style="whitegrid")

# 1. Histograms of each feature
df_scaled.hist(bins=30, figsize=(16, 12), edgecolor='black')
plt.suptitle("Histograms of Scaled Features", fontsize=20)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 2. Boxplots to examine spread and detect remaining outliers
plt.figure(figsize=(16, 10))
df_scaled.boxplot(rot=90)
plt.title("Boxplot of Scaled Features", fontsize=16)
plt.tight_layout()
plt.show()

# 3. Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df_scaled.corr(), annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title("Correlation Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

# 4. PCA for 2D projection
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_scaled)

# Optional: use 'Award?' column to color the clusters if available
if 'Award?' in df_no_outliers.columns:
    labels = df_no_outliers['Award?']
else:
    labels = None

plt.figure(figsize=(10, 8))
sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1], hue=labels, palette='Set2')
plt.title("PCA - 2D Projection", fontsize=16)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
if labels is not None:
    plt.legend(title="Award?")
plt.grid(True)
plt.tight_layout()
plt.show()

* Use multiple visualizations to understand the hidden patterns in the dataset

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1. Pair Plot (use only a few columns to avoid overload)
sample_columns = df_scaled[['Balance', 'Bonus_miles', 'Flight_miles_12mo', 'Days_since_enroll']]
sns.pairplot(sample_columns)
plt.suptitle("Pair Plot of Key Features", y=1.02)
plt.show()

# 2. Violin plots (comparing distribution by Award label if available)
if 'Award?' in df_no_outliers.columns:
    for col in ['Balance', 'Bonus_miles', 'Flight_miles_12mo']:
        plt.figure(figsize=(8, 4))
        sns.violinplot(x=df_no_outliers['Award?'], y=df_no_outliers[col])
        plt.title(f"Violin Plot of {col} by Award Status")
        plt.show()

# 3. t-SNE visualization for potential clusters
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(df_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=tsne_results[:, 0], y=tsne_results[:, 1], hue=df_no_outliers['Award?'], palette="coolwarm")
plt.title("t-SNE 2D Projection Colored by Award")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(title="Award?")
plt.grid(True)
plt.show()

# 4. Count plot for class distribution (e.g., Award?)
if 'Award?' in df_no_outliers.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(x='Award?', data=df_no_outliers)
    plt.title("Award Distribution")
    plt.show()

### **Implementing Clustering Algorithms:**

* Implement the K-Means, hierarchical, and DBSCAN algorithms using a programming language such as Python with libraries like scikit-learn or MATLAB.

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------
# 1. K-Means Clustering
# ----------------------------------------
inertia = []
silhouette = []

# Find optimal number of clusters using Elbow & Silhouette
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(df_scaled, kmeans.labels_))

# Plot Elbow and Silhouette
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(2, 11), inertia, marker='o')
plt.title('K-Means Elbow Method')
plt.xlabel('No. of Clusters')
plt.ylabel('Inertia')

plt.subplot(1, 2, 2)
plt.plot(range(2, 11), silhouette, marker='o', color='green')
plt.title('K-Means Silhouette Scores')
plt.xlabel('No. of Clusters')
plt.ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

# Final KMeans clustering (e.g., k=4)
kmeans_final = KMeans(n_clusters=4, random_state=42)
df_scaled['KMeans_Cluster'] = kmeans_final.fit_predict(df_scaled)

# ----------------------------------------
# 2. Hierarchical Clustering
# ----------------------------------------
# Generate linkage matrix
linkage_matrix = linkage(df_scaled.drop(columns=['KMeans_Cluster']), method='ward')

# Plot dendrogram
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
plt.title("Hierarchical Clustering Dendrogram (truncated)")
plt.xlabel("Sample Index or (Cluster Size)")
plt.ylabel("Distance")
plt.show()

# Assign cluster labels (e.g., 4 clusters)
df_scaled['Hierarchical_Cluster'] = fcluster(linkage_matrix, 4, criterion='maxclust')

# ----------------------------------------
# 3. DBSCAN
# ----------------------------------------
dbscan = DBSCAN(eps=1.5, min_samples=5)
df_scaled['DBSCAN_Cluster'] = dbscan.fit_predict(df_scaled)

# ----------------------------------------
# Visualize Clustering Results using PCA
# ----------------------------------------
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
components = pca.fit_transform(df_scaled.drop(columns=['KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster']))

def plot_clusters(column, title):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=df_scaled[column], palette='tab10', legend='full')
    plt.title(title)
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')
    plt.legend(title=column)
    plt.grid(True)
    plt.show()

plot_clusters('KMeans_Cluster', "K-Means Clustering (PCA Projection)")
plot_clusters('Hierarchical_Cluster', "Hierarchical Clustering (PCA Projection)")
plot_clusters('DBSCAN_Cluster', "DBSCAN Clustering (PCA Projection)")

* Apply each clustering algorithm to the pre-processed dataset to identify clusters within the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

# ========== K-Means ==========
kmeans = KMeans(n_clusters=4, random_state=42)
df_scaled['KMeans_Cluster'] = kmeans.fit_predict(df_scaled)

# ========== Hierarchical Clustering ==========
linked = linkage(df_scaled.drop(columns=['KMeans_Cluster']), method='ward')
df_scaled['Hierarchical_Cluster'] = fcluster(linked, 4, criterion='maxclust')

# ========== DBSCAN ==========
dbscan = DBSCAN(eps=1.5, min_samples=5)
df_scaled['DBSCAN_Cluster'] = dbscan.fit_predict(df_scaled)

# ========== PCA for Visualization ==========
pca = PCA(n_components=2)
pca_components = pca.fit_transform(df_scaled.drop(columns=['KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster']))

# Visualization Function
def plot_clusters(cluster_column, title):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=pca_components[:, 0], y=pca_components[:, 1], hue=df_scaled[cluster_column], palette='tab10')
    plt.title(title)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend(title=cluster_column)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# ========== Plot Results ==========
plot_clusters('KMeans_Cluster', 'K-Means Clusters (PCA View)')
plot_clusters('Hierarchical_Cluster', 'Hierarchical Clusters (PCA View)')
plot_clusters('DBSCAN_Cluster', 'DBSCAN Clusters (PCA View)')

* Experiment with different parameter settings for hierarchical clustering (e.g., linkage criteria), K-means (Elbow curve for different K values) and DBSCAN (e.g., epsilon, minPts) and evaluate the clustering results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.decomposition import PCA
import numpy as np

# ---------------------------
# K-MEANS: Elbow + Silhouette
# ---------------------------
inertia = []
silhouette = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(df_scaled.drop(columns=['KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster'], errors='ignore'))
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(df_scaled.drop(columns=['KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster'], errors='ignore'), labels))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(K_range, inertia, marker='o')
plt.title('K-Means Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')

plt.subplot(1, 2, 2)
plt.plot(K_range, silhouette, marker='o', color='green')
plt.title('K-Means Silhouette Score')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

# ---------------------------
# HIERARCHICAL: Linkage Methods
# ---------------------------
linkage_methods = ['ward', 'complete', 'average', 'single']
silhouette_scores_hier = []

for method in linkage_methods:
    linked = linkage(df_scaled, method=method)
    labels = fcluster(linked, 4, criterion='maxclust')
    sil = silhouette_score(df_scaled, labels)
    silhouette_scores_hier.append(sil)

plt.figure(figsize=(8, 5))
sns.barplot(x=linkage_methods, y=silhouette_scores_hier, palette='viridis')
plt.title("Silhouette Scores for Linkage Methods (Hierarchical)")
plt.ylabel("Silhouette Score")
plt.xlabel("Linkage Method")
plt.show()

# ---------------------------
# DBSCAN: Varying eps & min_samples
# ---------------------------
eps_values = [0.5, 1.0, 1.5, 2.0]
min_samples_values = [3, 5, 10]
results = []

for eps in eps_values:
    for min_samples in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(df_scaled)
        # Only evaluate if more than 1 cluster
        if len(set(labels)) > 1 and -1 in labels:
            sil = silhouette_score(df_scaled, labels)
            results.append((eps, min_samples, sil))
        else:
            results.append((eps, min_samples, -1))

### **Cluster Analysis & Interpretation:**

* Analyse the clusters generated by each clustering algorithm and interpret the characteristics of each cluster. Write you insights in few comments.

In [ ]:
# Restore the original (unscaled) data for interpretation
original_df = df_no_outliers.copy()
original_df.reset_index(drop=True, inplace=True)

# Append cluster labels to the original data
original_df['KMeans_Cluster'] = df_scaled['KMeans_Cluster']
original_df['Hierarchical_Cluster'] = df_scaled['Hierarchical_Cluster']
original_df['DBSCAN_Cluster'] = df_scaled['DBSCAN_Cluster']

# Function to summarize and interpret clusters
def summarize_clusters(df, cluster_col):
    print(f"\n===== {cluster_col} Summary =====")
    cluster_summary = df.groupby(cluster_col).mean().round(2)
    cluster_sizes = df[cluster_col].value_counts().sort_index()
    print(f"\nCluster Sizes:\n{cluster_sizes}")
    print(f"\nCluster Means:\n{cluster_summary}")
    return cluster_summary

# Analyze K-Means Clusters
kmeans_summary = summarize_clusters(original_df, 'KMeans_Cluster')

# Analyze Hierarchical Clusters
hier_summary = summarize_clusters(original_df, 'Hierarchical_Cluster')

# Analyze DBSCAN Clusters (excluding noise)
if -1 in original_df['DBSCAN_Cluster'].values:
    print("\nNote: DBSCAN cluster -1 = noise (outliers)")
dbscan_summary = summarize_clusters(original_df[original_df['DBSCAN_Cluster'] != -1], 'DBSCAN_Cluster')

# Optional: Display plots of each cluster center for visual inspection
import matplotlib.pyplot as plt

def plot_cluster_means(summary, title):
    summary.T.plot(kind='bar', figsize=(12, 6))
    plt.title(title)
    plt.ylabel('Average Value')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_cluster_means(kmeans_summary, "K-Means Cluster Profiles")
plot_cluster_means(hier_summary, "Hierarchical Cluster Profiles")
plot_cluster_means(dbscan_summary, "DBSCAN Cluster Profiles (Excl. Noise)")

### **Visualizations:**

* Visualize the clustering results using scatter plots or other suitable visualization techniques.
Plot the clusters with different colours to visualize the separation of data points belonging to different clusters.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Prepare PCA components from scaled data
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_scaled.drop(columns=['KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster']))

# Add PCA results for plotting
df_plot = df_scaled.copy()
df_plot['PCA1'] = pca_result[:, 0]
df_plot['PCA2'] = pca_result[:, 1]

# Plot function for each algorithm
def plot_clusters(data, cluster_col, title):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=data, x='PCA1', y='PCA2', hue=cluster_col, palette='tab10', s=60)
    plt.title(title)
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.legend(title=cluster_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Plot KMeans clusters
plot_clusters(df_plot, 'KMeans_Cluster', 'K-Means Clustering (PCA View)')

# Plot Hierarchical clusters
plot_clusters(df_plot, 'Hierarchical_Cluster', 'Hierarchical Clustering (PCA View)')

# Plot DBSCAN clusters (noise = -1 will appear as separate)
plot_clusters(df_plot, 'DBSCAN_Cluster', 'DBSCAN Clustering (PCA View)')

### **Evaluation and Performance Metrics:**

* Evaluate the quality of clustering using internal evaluation metrics such as silhouette score for K-Means and DBSCAN.

In [ ]:
from sklearn.metrics import silhouette_score

# Get features (excluding cluster labels)
features = df_scaled.drop(columns=['KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster'], errors='ignore')

# Silhouette Score for K-Means
kmeans_labels = df_scaled['KMeans_Cluster']
sil_kmeans = silhouette_score(features, kmeans_labels)
print(f"Silhouette Score for K-Means: {sil_kmeans:.4f}")

# Silhouette Score for DBSCAN
dbscan_labels = df_scaled['DBSCAN_Cluster']
# Check if DBSCAN found more than 1 cluster (excluding noise)
if len(set(dbscan_labels)) > 1 and -1 in dbscan_labels:
    sil_dbscan = silhouette_score(features, dbscan_labels)
    print(f"Silhouette Score for DBSCAN: {sil_dbscan:.4f}")
else:
    print("DBSCAN did not find enough clusters to compute Silhouette Score.")